## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para disponer de persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Bajar datasets si hace falta

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo Regresion Lineal

## 1.1 Init Experimento

In [3]:
# instalacion de paquetes que NO vienen por default en Colab
!pip install uv
!uv pip install -q kaggle
!uv pip install -q statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 73.4 MB/s eta 0:00:00


In [4]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [5]:
import os as os
import numpy as np
import polars as pl
import polars.selectors as cs
import statsmodels.api as sm

import warnings
warnings.filterwarnings("ignore")

Por favor, cargar aqui SU semilla primigenia

In [6]:
# defino los parametros
PARAM = {'experimento':'LR01',
  'kaggle_competition':'labo-iii-2026-rosario',
  'semilla_primigenia':102191,
  'empiojar_ruido':0.0
}

In [7]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/LR01


## 1.2 Preprocesamiento

In [8]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [9]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [10]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [11]:
display( tb_ventas )

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21276,201908,0.01265
21276,201909,0.01856
21276,201910,0.02079


### 1.2.1 Empiojado  lognormal

In [12]:

if PARAM['empiojar_ruido']>0.0:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=PARAM['empiojar_ruido'], size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


### 1.2.2  Dataset aplanado con lags

In [13]:
lags = [-2, *range(0,12)]

tb_lags = (
    tb_ventas.sort(["product_id", "periodo"])
      .with_columns(
          [
              pl.col("tn")
                .shift(lag)
                .over("product_id")
                .alias(f"tn_{lag}")
              for lag in lags
          ]
      )
)

tb_lags = tb_lags.rename({"tn_-2": "clase"})

In [14]:
display(tb_lags)

product_id,periodo,tn,clase,tn_0,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20001,201701,934.77222,1303.35771,934.77222,null,null,null,null,null,null,null,null,null,null,null
20001,201702,798.0162,1069.9613,798.0162,934.77222,null,null,null,null,null,null,null,null,null,null
20001,201703,1303.35771,1502.20132,1303.35771,798.0162,934.77222,null,null,null,null,null,null,null,null,null
20001,201704,1069.9613,1520.06539,1069.9613,1303.35771,798.0162,934.77222,null,null,null,null,null,null,null,null
20001,201705,1502.20132,1030.67391,1502.20132,1069.9613,1303.35771,798.0162,934.77222,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
21276,201908,0.01265,0.02079,0.01265,0.00223,0.04086,0.09283,0.10173,0.12249,null,null,null,null,null,null
21276,201909,0.01856,0.03341,0.01856,0.01265,0.00223,0.04086,0.09283,0.10173,0.12249,null,null,null,null,null
21276,201910,0.02079,0.00892,0.02079,0.01856,0.01265,0.00223,0.04086,0.09283,0.10173,0.12249,null,null,null,null


### 1.2.4  Definicion de Training

In [15]:
# Esta es la salsa mágica del notebook

productos_magicos = [ 20002, 20003, 20005, 20006, 20009, 20010, 20011, 20013, 20015,
  20017, 20018, 20019, 20021, 20022, 20026, 20028, 20035, 20038, 20042, 20043, 20044,
  20045, 20046, 20049, 20051, 20052, 20055, 20081, 20324, 20417, 20771, 20947, 20949, 21003
]

productos_magicos = [ 20001, 20002, 20005, 20013, 20033, 20037, 20038, 20043, 20044,
  20045, 20046, 20052, 20055, 20058, 20059, 20069, 20070, 20072, 20073, 20075, 20080,
  20091, 20094, 20099, 20107, 20114, 20120, 20132, 20137, 20139, 20142, 20144, 20146,
  20148, 20151, 20153, 20157, 20158, 20161, 20162, 20166, 20167, 20189, 20198, 20201,
  20202, 20203, 20208, 20226, 20228, 20231, 20233, 20253, 20254, 20256, 20269, 20270,
  20271, 20275, 20276, 20277, 20278, 20288, 20298, 20315, 20317, 20320, 20322, 20335,
  20337, 20338, 20344, 20348, 20350, 20353, 20359, 20385, 20390, 20398, 20402, 20403,
  20406, 20411, 20416, 20417, 20418, 20419, 20421, 20422, 20424, 20428, 20429, 20443,
  20456, 20466, 20469, 20479, 20497, 20500, 20509, 20514, 20517, 20524, 20532, 20549,
  20551, 20560, 20561, 20565, 20568, 20579, 20583, 20585, 20586, 20589, 20599, 20606,
  20614, 20624, 20632, 20642, 20646, 20653, 20655, 20657, 20660, 20661, 20663, 20666,
  20677, 20680, 20684, 20696, 20699, 20713, 20737, 20744, 20745, 20765, 20768, 20773,
  20777, 20786, 20789, 20800, 20807, 20812, 20818, 20830, 20832, 20838, 20847, 20855,
  20863, 20864, 20882, 20883, 20906, 20913, 20914, 20919, 20922, 20925, 20937, 20945,
  20956, 20961, 20965, 20970, 20976, 20986, 20996, 21016, 21038, 21048, 21049, 21077,
  21080, 21088, 21118, 21170, 21200
]

In [16]:
# Entreno con los datos de 2018 para los  productos_magicos

dtrain = tb_lags.filter( (pl.col("periodo") == 201812) & (pl.col("product_id").is_in(productos_magicos)) )

display( dtrain )

product_id,periodo,tn,clase,tn_0,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20001,201812,1486.68669,1259.09363,1486.68669,1813.01511,2295.19832,1438.67455,1800.96168,1470.41009,1150.79169,1293.89788,1251.28462,1856.83534,1043.7647,1169.07532
20002,201812,1009.45458,1043.01349,1009.45458,1766.81068,1378.49032,954.23575,1161.8843,977.40239,1033.82845,1103.39191,999.20934,966.86044,712.00087,984.80167
20005,201812,372.63428,409.8995,372.63428,469.26344,893.74086,761.7752,874.88924,502.34077,547.62513,637.11135,496.41774,559.98671,399.20878,417.53208
20013,201812,333.70155,377.10855,333.70155,367.82928,469.93401,235.49526,437.85378,391.482,432.0225,494.79885,448.49259,593.35731,382.2273,329.57379
20033,201812,150.48852,132.55515,150.48852,253.0437,278.24706,300.47745,266.28693,207.81852,175.15407,246.47805,190.81062,228.00414,163.39323,171.61053
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
21080,201812,0.41052,0.1834,0.41052,0.39856,0.30683,0.42696,0.5951,0.6628,0.62566,0.50007,0.76106,0.54048,0.27629,0.42806
21088,201812,0.22682,0.49756,0.22682,0.4234,0.33713,0.55409,0.51011,0.50419,0.81655,0.80695,0.64965,0.91654,0.49415,0.37283
21118,201812,0.31089,0.4323,0.31089,0.27071,0.32891,0.38184,0.46436,0.48293,0.46226,0.67321,0.48212,0.75515,0.52254,0.76033


## 1.3  Modelo de Regresion Lineal

In [17]:

# campos a utilizar
campos_buenos = dtrain.select(cs.starts_with("tn_"))


# tristemente paso por pandas
X_train = dtrain.select(campos_buenos).to_pandas()

# artificio para agregar intercepto
X_train = sm.add_constant(X_train)

# tristemente paso por pandas
y_train = dtrain['clase'].to_pandas()


modelo = sm.OLS(y_train, X_train).fit()
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                  clase   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.989
Method:                 Least Squares   F-statistic:                     1336.
Date:                Fri, 05 Jun 2026   Prob (F-statistic):          1.41e-160
Time:                        17:41:01   Log-Likelihood:                -730.19
No. Observations:                 182   AIC:                             1486.
Df Residuals:                     169   BIC:                             1528.
Df Model:                          12                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0706      1.259      0.056      0.9

## 1.4 Aplicacion a los datos del futuro

### 1.4.1  Definicion datos del futuro
Solo puedo aplicar al modelo a los datos que tengan completos todos los meses
<br> de los 780 productos a predecir
* 656 tienen todos los meses del 2019 completos
* 124 no (los voy a predecir con el simple promedio)

In [18]:
dfuture = tb_lags.filter( (pl.col("periodo") == 201912) & (pl.col("tn_11").is_not_null() ))

display( dfuture )

product_id,periodo,tn,clase,tn_0,tn_1,tn_2,tn_3,tn_4,tn_5,tn_6,tn_7,tn_8,tn_9,tn_10,tn_11
i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
20001,201912,1504.68856,null,1504.68856,1397.37231,1561.50552,1660.00561,1261.34529,1678.99318,1109.93769,1629.78233,1647.63848,1470.65653,1259.09363,1275.77351
20002,201912,1087.30855,null,1087.30855,1423.57739,1979.53635,1090.18771,813.78215,1066.44999,928.36431,1034.98927,1287.62346,1083.62552,1043.01349,1266.78751
20003,201912,892.50129,null,892.50129,948.29393,1081.36645,967.77116,635.59563,715.20314,662.38654,590.12515,565.33774,638.0401,758.32657,964.76919
20004,201912,637.90002,null,637.90002,723.94206,1064.69633,786.1714,482.13372,521.71519,667.19411,603.31081,466.70901,619.77084,441.70332,511.33713
20005,201912,593.24443,null,593.24443,606.91173,996.78275,879.52808,536.668,745.74978,876.39696,897.26297,624.9988,488.21387,409.8995,363.58438
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
21248,201912,0.01129,null,0.01129,0.02964,0.0127,0.01411,0.02117,0.02116,0.00988,0.01553,0.03106,0.05365,0.06209,0.02962
21256,201912,0.01271,null,0.01271,0.02682,0.00847,0.00423,0.02965,0.02822,0.00988,0.01553,0.01835,0.0593,0.05081,0.03811
21259,201912,0.01412,null,0.01412,0.02965,0.01975,0.00564,0.03106,0.04657,0.00988,0.01976,0.02117,0.06777,0.0508,0.04234


### 1.4.2  Aplicacion del modelo de regresion a los datos del futuro

Se aplica el modelo a los 656 datos del futuro

In [19]:

# campos a utilizar
campos_buenos = dfuture.select(cs.starts_with("tn_"))

# tristemente paso por pandas
X_future = dfuture.select(campos_buenos).to_pandas()

# artificio para agregar intercepto
X_future = sm.add_constant(X_future)

# la prediccion
prediccion = modelo.predict(X_future)


tb_regresion = dfuture.select(['product_id']).with_columns(
    pl.Series("tn_pred", prediccion)
)

display( tb_regresion)

product_id,tn_pred
i64,f64
20001,1124.297853
20002,1111.127719
20003,735.277478
20004,575.581884
20005,504.557617
…,…
21248,0.106491
21256,0.104122
21259,0.105241


### 1.4.3  Join de los modelos de regresion y promedio

In [20]:
# tabla conh los promedios
primer_periodo = 201901
ultimo_periodo = 201912
tb_meses12 = tb_ventas.filter( pl.col("periodo").is_between(primer_periodo,ultimo_periodo)).group_by("product_id").agg(
 pl.col("tn").mean().alias("tn"))

tb_meses12 = tb_meses12.select(["product_id", "tn"])
display( tb_meses12 )

product_id,tn
i64,f64
20001,1454.73272
20002,1175.437142
20003,784.976408
20004,627.215328
20005,668.270104
…,…
21263,0.029993
21265,0.089541
21266,0.094659


In [21]:
# Update table1 using table2
tb_final = (
    tb_meses12
    .join(tb_regresion, on="product_id", how="left", suffix="_update")
    .with_columns(
        pl.coalesce([pl.col("tn_pred"), pl.col("tn")]).alias("tn")
    )
    .drop("tn_pred")
)

display( tb_final )

product_id,tn
i64,f64
20001,1124.297853
20002,1111.127719
20003,735.277478
20004,575.581884
20005,504.557617
…,…
21263,0.108509
21265,0.089541
21266,0.094659


## 1.5 Submit a Kaggle

In [22]:
# Submit a Kaggle
if PARAM['empiojar_ruido']<=0.0:
  archivo= "linreg.csv"
  mensaje= "Regresion Lineal"
else:
  archivo= "linreg_empiojado.csv"
  mensaje= "Linear Regression logEMPIOJADO al " + str(PARAM['empiojar_ruido'])

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )

[![Ver video](https://www.youtube.com/embed/IY9G79BDRS4)](https://www.youtube.com/embed/IY9G79BDRS4)

In [28]:
print(modelo.params)

const    0.070573
tn_0     0.040255
tn_1     0.235318
tn_2     0.051261
tn_3    -0.071108
tn_4    -0.093344
tn_5    -0.121505
tn_6     0.275105
tn_7    -0.009183
tn_8     0.007269
tn_9     0.201441
tn_10    0.251542
tn_11    0.140497
dtype: float64


In [31]:
import polars as pl

# Calculate range for productos_magicos, excluding 0 'toneladas'
df_magicos = tb_ventas.filter(pl.col("product_id").is_in(productos_magicos))
min_tn_magicos = df_magicos.filter(pl.col("tn") > 0).select(pl.col("tn").min()).item()
max_tn_magicos = df_magicos.select(pl.col("tn").max()).item()

# Calculate range for products not in productos_magicos but within the same product_id bounds, excluding 0 'toneladas'
min_product_id_overall = min(productos_magicos)
max_product_id_overall = max(productos_magicos)

df_non_magicos = tb_ventas.filter(
    (pl.col("product_id").is_between(min_product_id_overall, max_product_id_overall)) &
    (~pl.col("product_id").is_in(productos_magicos))
)
min_tn_non_magicos = df_non_magicos.filter(pl.col("tn") > 0).select(pl.col("tn").min()).item()
max_tn_non_magicos = df_non_magicos.select(pl.col("tn").max()).item()

print(f"Rango de Toneladas para productos_magicos (excluyendo 0):")
print(f"  Min: {min_tn_magicos}, Max: {max_tn_magicos:.2f}")
print(f"\nRango de Toneladas para productos NO mágicos (dentro del mismo rango de ID, excluyendo 0):")
print(f"  Min: {min_tn_non_magicos}, Max: {max_tn_non_magicos:.2f}")

Rango de Toneladas para productos_magicos (excluyendo 0):
  Min: 0.00092, Max: 2295.20

Rango de Toneladas para productos NO mágicos (dentro del mismo rango de ID, excluyendo 0):
  Min: 0.00089, Max: 1958.60


In [38]:
import plotly.express as px
import polars as pl

# Choose a product_id from productos_magicos to plot as an example
# Using the same example_product_id as the Matplotlib plot for consistency

# Filter for the example product and the last 12 months
# Using primer_periodo and ultimo_periodo from the notebook state for consistency
product_data_plotly = tb_ventas.filter(
    (pl.col("product_id") == example_product_id) &
    (pl.col("periodo").is_between(primer_periodo, ultimo_periodo))
).sort("periodo")

# Convert 'periodo' to string for proper categorical axis in Plotly if desired
product_data_plotly = product_data_plotly.with_columns(pl.col("periodo").cast(pl.Utf8).alias("periodo_str"))

if not product_data_plotly.is_empty():
    fig = px.line(
        product_data_plotly.to_pandas(), # Plotly generally works better with Pandas DataFrames
        x="periodo_str",
        y="tn",
        title=f'Toneladas por Periodo para Product ID: {example_product_id} (últimos 12 meses) - Plotly',
        labels={
            "periodo_str": "Periodo",
            "tn": "Toneladas (tn)"
        },
        line_shape="linear",
        markers=True
    )

    fig.update_layout(
        hovermode="x unified",
        xaxis_tickangle=-45
    )

    fig.show()
else:
    print(f"No data found for Product ID: {example_product_id} in the specified period for Plotly visualization.")

In [35]:
import plotly.express as px
import polars as pl

# Choose a product_id from productos_magicos to plot as an example
# Using the same example_product_id as the Matplotlib plot for consistency

# Filter for the example product and the last 12 months
# Using primer_periodo and ultimo_periodo from the notebook state for consistency
product_data_plotly = tb_ventas.filter(
    (pl.col("product_id") == example_product_id) &
    (pl.col("periodo").is_between(primer_periodo, ultimo_periodo))
).sort("periodo")

# Convert 'periodo' to string for proper categorical axis in Plotly if desired
product_data_plotly = product_data_plotly.with_columns(pl.col("periodo").cast(pl.Utf8).alias("periodo_str"))

if not product_data_plotly.is_empty():
    fig = px.line(
        product_data_plotly.to_pandas(), # Plotly generally works better with Pandas DataFrames
        x="periodo_str",
        y="tn",
        title=f'Toneladas por Periodo para Product ID: {example_product_id} (últimos 12 meses) - Plotly',
        labels={
            "periodo_str": "Periodo",
            "tn": "Toneladas (tn)"
        },
        line_shape="linear",
        markers=True
    )

    fig.update_layout(
        hovermode="x unified",
        xaxis_tickangle=-45
    )

    fig.show()
else:
    print(f"No data found for Product ID: {example_product_id} in the specified period for Plotly visualization.")

In [36]:
import plotly.express as px
import polars as pl

# Filter tb_ventas for products within the overall min/max product_id range and the last 12 months
# Using primer_periodo, ultimo_periodo, min_product_id_overall, and max_product_id_overall from the notebook state
df_filtered_range_plotly = tb_ventas.filter(
    (pl.col("product_id").is_between(min_product_id_overall, max_product_id_overall)) &
    (pl.col("periodo").is_between(primer_periodo, ultimo_periodo))
).sort(["product_id", "periodo"])

# Convert 'periodo' to string for proper categorical axis in Plotly
df_filtered_range_plotly = df_filtered_range_plotly.with_columns(pl.col("periodo").cast(pl.Utf8).alias("periodo_str"))

if not df_filtered_range_plotly.is_empty():
    fig = px.line(
        df_filtered_range_plotly.to_pandas(),
        x="periodo_str",
        y="tn",
        color="product_id",
        title=f'Toneladas por Periodo para Productos en Rango [{min_product_id_overall}-{max_product_id_overall}] (últimos 12 meses) - Plotly',
        labels={
            "periodo_str": "Periodo",
            "tn": "Toneladas (tn)",
            "product_id": "ID del Producto"
        },
        line_shape="linear",
        markers=True
    )

    fig.update_layout(
        hovermode="x unified",
        xaxis_tickangle=-45
    )

    fig.show()
else:
    print(f"No data found for products in the range [{min_product_id_overall}-{max_product_id_overall}] in the specified period for Plotly visualization.")

In [37]:
import plotly.express as px
import polars as pl

# Filter tb_ventas for products within the overall min/max product_id range BUT EXCLUDING magical products
# Using primer_periodo, ultimo_periodo, min_product_id_overall, max_product_id_overall, and productos_magicos from the notebook state
df_filtered_range_excluded_plotly = tb_ventas.filter(
    (pl.col("product_id").is_between(min_product_id_overall, max_product_id_overall)) &
    (~pl.col("product_id").is_in(productos_magicos)) &
    (pl.col("periodo").is_between(primer_periodo, ultimo_periodo))
).sort(["product_id", "periodo"])

# Convert 'periodo' to string for proper categorical axis in Plotly
df_filtered_range_excluded_plotly = df_filtered_range_excluded_plotly.with_columns(pl.col("periodo").cast(pl.Utf8).alias("periodo_str"))

if not df_filtered_range_excluded_plotly.is_empty():
    fig = px.line(
        df_filtered_range_excluded_plotly.to_pandas(),
        x="periodo_str",
        y="tn",
        color="product_id",
        title=f'Toneladas por Periodo para Productos NO Mágicos en Rango [{min_product_id_overall}-{max_product_id_overall}] (últimos 12 meses) - Plotly',
        labels={
            "periodo_str": "Periodo",
            "tn": "Toneladas (tn)",
            "product_id": "ID del Producto"
        },
        line_shape="linear",
        markers=True
    )

    fig.update_layout(
        hovermode="x unified",
        xaxis_tickangle=-45
    )

    fig.show()
else:
    print(f"No data found for non-magical products in the range [{min_product_id_overall}-{max_product_id_overall}] in the specified period for Plotly visualization.")

In [39]:
import plotly.express as px
import polars as pl

# Filter tb_ventas for the productos_magicos and the last 12 months
df_filtered_magicos_plotly = tb_ventas.filter(
    (pl.col("product_id").is_in(productos_magicos)) &
    (pl.col("periodo").is_between(primer_periodo, ultimo_periodo))
).sort(["product_id", "periodo"])

# Convert 'periodo' to string for proper categorical axis in Plotly if desired, or keep as numeric
df_filtered_magicos_plotly = df_filtered_magicos_plotly.with_columns(pl.col("periodo").cast(pl.Utf8).alias("periodo_str"))

if not df_filtered_magicos_plotly.is_empty():
    fig = px.line(
        df_filtered_magicos_plotly.to_pandas(), # Plotly generally works better with Pandas DataFrames
        x="periodo_str",
        y="tn",
        color="product_id",
        title='Toneladas por Periodo para Productos Mágicos (últimos 12 meses)',
        labels={
            "periodo_str": "Periodo",
            "tn": "Toneladas (tn)",
            "product_id": "ID del Producto"
        },
        line_shape="linear", # 'linear', 'spline', 'hv', 'vh', 'hvh'
        markers=True
    )

    fig.update_layout(
        hovermode="x unified", # Shows all traces on hover
        xaxis_tickangle=-45 # Rotate x-axis labels
    )

    fig.show()
else:
    print("No data found for the specified magical products in the last 12 months for Plotly visualization.")

In [34]:
import plotly.express as px
import polars as pl

# Filter tb_ventas for the productos_magicos and the last 12 months
df_filtered_magicos_plotly = tb_ventas.filter(
    (pl.col("product_id").is_in(productos_magicos)) &
    (pl.col("periodo").is_between(primer_periodo, ultimo_periodo))
).sort(["product_id", "periodo"])

# Convert 'periodo' to string for proper categorical axis in Plotly if desired, or keep as numeric
df_filtered_magicos_plotly = df_filtered_magicos_plotly.with_columns(pl.col("periodo").cast(pl.Utf8).alias("periodo_str"))

if not df_filtered_magicos_plotly.is_empty():
    fig = px.line(
        df_filtered_magicos_plotly.to_pandas(), # Plotly generally works better with Pandas DataFrames
        x="periodo_str",
        y="tn",
        color="product_id",
        title='Toneladas por Periodo para Productos Mágicos (últimos 12 meses)',
        labels={
            "periodo_str": "Periodo",
            "tn": "Toneladas (tn)",
            "product_id": "ID del Producto"
        },
        line_shape="linear", # 'linear', 'spline', 'hv', 'vh', 'hvh'
        markers=True
    )

    fig.update_layout(
        hovermode="x unified", # Shows all traces on hover
        xaxis_tickangle=-45 # Rotate x-axis labels
    )

    fig.show()
else:
    print("No data found for the specified magical products in the last 12 months for Plotly visualization.")